# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [20]:
df = pd.read_csv("data/AviationData.csv", encoding="latin1") 

df.head()
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

/var/folders/b1/d60q06cx5s92hfrk5_88lks80000gn/T/ipykernel_34243/2722263948.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="latin1")


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [21]:
df = df[df["Aircraft.Category"] == "Airplane"]
df = df[df["Amateur.Built"] == "No"]

df.head()


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
5,20170710X52551,Accident,NYC79AA106,1979-09-17,"BOSTON, MA",United States,42.445277,-70.758333,NaN,NaN,...,NaN,Air Canada,NaN,NaN,1.0,44.0,VMC,Climb,Probable Cause,19-09-2017
7,20020909X01562,Accident,SEA82DA022,1982-01-01,"PULLMAN, WA",United States,NaN,NaN,NaN,BLACKBURN AG STRIP,...,Personal,NaN,0.0,0.0,0.0,2.0,VMC,Takeoff,Probable Cause,01-01-1982
8,20020909X01561,Accident,NYC82DA015,1982-01-01,"EAST HANOVER, NJ",United States,NaN,NaN,N58,HANOVER,...,Business,NaN,0.0,0.0,0.0,2.0,IMC,Landing,Probable Cause,01-01-1982
12,20020917X02148,Accident,FTW82FRJ07,1982-01-02,"HOMER, LA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,0.0,0.0,1.0,0.0,IMC,Cruise,Probable Cause,02-01-1983
13,20020917X02134,Accident,FTW82FRA14,1982-01-02,"HEARNE, TX",United States,NaN,NaN,T72,HEARNE MUNICIPAL,...,Personal,NaN,1.0,0.0,0.0,0.0,IMC,Takeoff,Probable Cause,02-01-1983


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [22]:
# Make injury columns numeric
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing injury counts with 0
df[injury_cols] = df[injury_cols].fillna(0)

# Create total serious/fatal injuries column
df["Total.Fatal.Serious"] = (
    df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
)

# Estimate total passengers / people onboard
df["Total.Onboard"] = (
    df["Total.Fatal.Injuries"]
    + df["Total.Serious.Injuries"]
    + df["Total.Minor.Injuries"]
    + df["Total.Uninjured"]
)

# Avoid division by zero by replacing 0 with NaN
df["Total.Onboard"] = df["Total.Onboard"].replace(0, np.nan)

# Create fraction of passengers seriously/fatally injured
df["Fatal.Serious.Fraction"] = df["Total.Fatal.Serious"] / df["Total.Onboard"]

# Quick check
df[[
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Total.Fatal.Serious",
    "Total.Onboard",
    "Fatal.Serious.Fraction"
]].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.Fatal.Serious,Total.Onboard,Fatal.Serious.Fraction
5,0.0,0.0,1.0,44.0,0.0,45.0,0.0
7,0.0,0.0,0.0,2.0,0.0,2.0,0.0
8,0.0,0.0,0.0,2.0,0.0,2.0,0.0
12,0.0,0.0,1.0,0.0,0.0,1.0,0.0
13,1.0,0.0,0.0,0.0,1.0,1.0,1.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [23]:
# Inspect damage values
df["Aircraft.damage"].value_counts(dropna=False)

# Standardize text
df["Aircraft.damage"] = df["Aircraft.damage"].str.strip().str.title()

# Create binary destroyed column
df["Destroyed"] = np.where(df["Aircraft.damage"] == "Destroyed", 1, 0)

# If damage is missing, make Destroyed NaN instead of assuming 0
df.loc[df["Aircraft.damage"].isna(), "Destroyed"] = np.nan

# Quick check
df[["Aircraft.damage", "Destroyed"]].head()

,Aircraft.damage,Destroyed
5,Substantial,0.0
7,Substantial,0.0
8,Substantial,0.0
12,Destroyed,1.0
13,Destroyed,1.0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [24]:
# Inspect make values
df["Make"].value_counts().head(20)

# Standardize formatting
df["Make"] = df["Make"].str.strip().str.upper()

# Check again
df["Make"].value_counts().head(20)

# Keep only makes with enough data points (>= 50 accidents)
make_counts = df["Make"].value_counts()

valid_makes = make_counts[make_counts >= 50].index

df = df[df["Make"].isin(valid_makes)]

# Check results
df["Make"].value_counts().head()

Make
CESSNA    8443
PIPER     4700
BEECH     1684
BOEING    1309
MOONEY     417
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [25]:
# Remove missing model values
df = df.dropna(subset=["Model"])

# Standardize model formatting
df["Model"] = df["Model"].str.strip().str.upper()

# Create a unique plane identifier
df["Plane.Type"] = df["Make"] + " " + df["Model"]

# Inspect most common plane types
df["Plane.Type"].value_counts().head(20)

Plane.Type
CESSNA 172                 866
CESSNA 152                 448
BOEING 737                 403
CESSNA 182                 344
CESSNA 172N                314
CESSNA 172S                276
PIPER PA28                 273
CESSNA 150                 255
CESSNA 180                 237
PIPER PA-28-140            229
CESSNA 172M                217
PIPER PA-18-150            196
BEECH A36                  175
CESSNA 172P                167
CIRRUS DESIGN CORP SR22    144
PIPER PA-28-161            140
CESSNA 140                 132
PIPER PA-28-180            129
CESSNA 170B                122
CESSNA 210                 118
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [26]:
# Inspect columns related to accident conditions

df["Engine.Type"].value_counts(dropna=False)

df["Weather.Condition"].value_counts(dropna=False)

df["Number.of.Engines"].value_counts(dropna=False)

df["Purpose.of.flight"].value_counts(dropna=False)

df["Broad.phase.of.flight"].value_counts(dropna=False)


# Standardize formatting (remove spaces and unify case)
df["Engine.Type"] = df["Engine.Type"].str.strip().str.upper()

df["Weather.Condition"] = df["Weather.Condition"].str.strip().str.upper()

df["Purpose.of.flight"] = df["Purpose.of.flight"].str.strip().str.upper()

df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].str.strip().str.upper()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [27]:
# Check number of non-null values per column
df.info()

# Keep only columns with at least 20,000 non-null values
df = df.dropna(axis=1, thresh=20000)

# Check remaining columns
df.info()

<class 'pandas.DataFrame'>
Index: 20764 entries, 5 to 88886
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                20764 non-null  str    
 1   Investigation.Type      20764 non-null  str    
 2   Accident.Number         20764 non-null  str    
 3   Event.Date              20764 non-null  str    
 4   Location                20759 non-null  str    
 5   Country                 20757 non-null  str    
 6   Latitude                16072 non-null  object 
 7   Longitude               16069 non-null  object 
 8   Airport.Code            13126 non-null  str    
 9   Airport.Name            13614 non-null  str    
 10  Injury.Severity         20045 non-null  str    
 11  Aircraft.damage         19656 non-null  str    
 12  Aircraft.Category       20764 non-null  str    
 13  Registration.Number     20595 non-null  str    
 14  Make                    20764 non-null  str    
 15  M

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [28]:
# Save cleaned dataset for analysis notebook
df.to_csv("data/clean_aviation_accidents.csv", index=False)
